In [1]:
import torch.nn as nn
import pandas as pd
import torch
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
import platform

print("Torch has been imported")

Torch has been imported


In [2]:
def get_platform_device():
    '''

    Detects the OS and the best GPU to use on the device

    '''
    
    os_name = platform.system()
    processor = platform.processor()
    

    if torch.cuda.is_available():
        device = torch.device("cuda")
        platform_info = f"Nvidia GPU via CUDA ({os_name})"
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        platform_info = f"Apple Silicon via MPS ({os_name})"
    else:
        device = torch.device("cpu")
        platform_info = f"CPU fallback ({os_name} {processor})"
    
    print(f"Platform detected: {platform_info}")
    return device

device = get_platform_device()
print(f"Using device: {device}")

Platform detected: Apple Silicon via MPS (Darwin)
Using device: mps


In [3]:
FILEPATH = "data/asl_data/sign_mnist_valid.csv"
train_df = pd.read_csv(FILEPATH)
valid_df = pd.read_csv(FILEPATH)
print("Data loaded!")

Data loaded!


In [4]:
sample_df = train_df.head().copy()
sample_df.pop('label')
sample_x = sample_df.values
sample_x

array([[149, 149, 150, ..., 112, 120, 107],
       [126, 128, 131, ..., 184, 182, 180],
       [ 85,  88,  92, ..., 225, 224, 222],
       [203, 205, 207, ..., 240, 253, 255],
       [188, 191, 193, ...,  46,  46,  53]], shape=(5, 784))

In [5]:
sample_x.shape

(5, 784)

In [6]:
IMG_HEIGHT = 28
IMG_WIDTH = 28
IMG_CHS = 1

sample_x = sample_x.reshape(-1, IMG_CHS, IMG_HEIGHT, IMG_WIDTH)
sample_x.shape

(5, 1, 28, 28)

In [7]:
class MyDataset(Dataset):
    def __init__(self, base_df):
        x_df = base_df.copy()  # Some operations below are in-place
        y_df = x_df.pop('label')
        x_df = x_df.values / 255  # Normalize values from 0 to 1
        x_df = x_df.reshape(-1, IMG_CHS, IMG_WIDTH, IMG_HEIGHT)
        self.xs = torch.tensor(x_df).float().to(device)
        self.ys = torch.tensor(y_df).to(device)

    def __getitem__(self, idx):
        x = self.xs[idx]
        y = self.ys[idx]
        return x, y

    def __len__(self):
        return len(self.xs)

In [8]:
BATCH_SIZE = 32
train_data = MyDataset(train_df)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
train_N = len(train_loader.dataset)

valid_data = MyDataset(valid_df)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE)
valid_N = len(valid_loader.dataset)
print("Dataset complete")

Dataset complete


In [9]:
batch = next(iter(train_loader))
batch

[tensor([[[[0.6118, 0.6157, 0.6157,  ..., 0.6157, 0.6157, 0.6235],
           [0.6353, 0.6353, 0.6353,  ..., 0.6275, 0.6314, 0.6235],
           [0.6431, 0.6431, 0.6431,  ..., 0.6431, 0.6431, 0.6118],
           ...,
           [0.7451, 0.7961, 0.7922,  ..., 0.8118, 0.8078, 0.8000],
           [0.7608, 0.7804, 0.7569,  ..., 0.8196, 0.8118, 0.8078],
           [0.7608, 0.7686, 0.7725,  ..., 0.8235, 0.8196, 0.8157]]],
 
 
         [[[0.5098, 0.5176, 0.5216,  ..., 0.4510, 0.4431, 0.4392],
           [0.5176, 0.5176, 0.5255,  ..., 0.4549, 0.4471, 0.4392],
           [0.5137, 0.5176, 0.5216,  ..., 0.4549, 0.4510, 0.4471],
           ...,
           [0.6000, 0.6039, 0.6078,  ..., 0.5451, 0.5373, 0.5294],
           [0.6000, 0.6078, 0.6118,  ..., 0.5412, 0.5373, 0.5294],
           [0.6000, 0.6078, 0.6118,  ..., 0.5412, 0.5373, 0.5294]]],
 
 
         [[[0.4196, 0.4353, 0.4549,  ..., 0.3686, 0.5098, 0.6510],
           [0.4275, 0.4431, 0.4627,  ..., 0.4235, 0.5137, 0.6627],
           [0.4392

In [10]:
batch[0].shape

torch.Size([32, 1, 28, 28])

In [11]:
batch[1].shape

torch.Size([32])

In [ ]:
## Creating a model

In [12]:
n_classes = 24
kernel_size = 3
flattened_img_size = 75 * 3 * 3

model = nn.Sequential(
    # First convolution
    nn.Conv2d(IMG_CHS, 25, kernel_size, stride=1, padding=1),  # 25 x 28 x 28
    nn.BatchNorm2d(25),
    nn.ReLU(),
    nn.MaxPool2d(2, stride=2),  # 25 x 14 x 14
    # Second convolution
    nn.Conv2d(25, 50, kernel_size, stride=1, padding=1),  # 50 x 14 x 14
    nn.BatchNorm2d(50),
    nn.ReLU(),
    nn.Dropout(.2),
    nn.MaxPool2d(2, stride=2),  # 50 x 7 x 7
    # Third convolution
    nn.Conv2d(50, 75, kernel_size, stride=1, padding=1),  # 75 x 7 x 7
    nn.BatchNorm2d(75),
    nn.ReLU(),
    nn.MaxPool2d(2, stride=2),  # 75 x 3 x 3
    # Flatten to Dense
    nn.Flatten(),
    nn.Linear(flattened_img_size, 512),
    nn.Dropout(.3),
    nn.ReLU(),
    nn.Linear(512, n_classes)
)

backend = "aot_eager" if device.type == "mps" else "inductor"   # Inductor breaks sometimes
model = torch.compile(model.to(device), backend=backend)
    
model
loss_function = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters())

print(f"Using backend: {backend}")

Using backend: aot_eager


In [13]:
def get_batch_accuracy(output, y, N):
    pred = output.argmax(dim=1, keepdim=True)
    correct = pred.eq(y.view_as(pred)).sum().item()
    return correct / N

In [14]:
def validate():
    loss = 0
    accuracy = 0

    model.eval()
    with torch.no_grad():
        for x, y in valid_loader:
            output = model(x)

            loss += loss_function(output, y).item()
            accuracy += get_batch_accuracy(output, y, valid_N)
    print('Valid - Loss: {:.4f} Accuracy: {:.4f}'.format(loss, accuracy))

In [15]:
def train():
    loss = 0
    accuracy = 0

    model.train()
    for x, y in train_loader:
        output = model(x)
        optimizer.zero_grad()
        batch_loss = loss_function(output, y)
        batch_loss.backward()
        optimizer.step()

        loss += batch_loss.item()
        accuracy += get_batch_accuracy(output, y, train_N)
    print('Train - Loss: {:.4f} Accuracy: {:.4f}'.format(loss, accuracy))

In [16]:
epochs = 20
print(backend)
for epoch in range(epochs):
    print(f'Epoch: {epoch+1}')
    train()
    validate()
print("training complete")

aot_eager
Epoch: 1


/Users/billy/Code/Python/Lab/.venv/lib/python3.14/site-packages/torch/autograd/graph.py:869: UserWarning: Error detected in MaxPool2DBackward0. Traceback of forward call that caused the error:
 (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/python_anomaly_mode.cpp:127.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
W0428 12:35:48.996000 16188 torch/_dynamo/exc.py:623] [0/1] Backend compiler exception
W0428 12:35:48.996000 16188 torch/_dynamo/exc.py:623] [0/1]   Explanation: Backend compiler `aot_eager` failed with aten.max_pool2d_backward.default. Adding a graph break.
W0428 12:35:48.996000 16188 torch/_dynamo/exc.py:623] [0/1]   Hint: Report an issue to the backend compiler repo.
W0428 12:35:48.996000 16188 torch/_dynamo/exc.py:623] [0/1] 
W0428 12:35:48.996000 16188 torch/_dynamo/exc.py:623] [0/1]   Developer debug context: Backend: aot_eager
W0428 12:35:48.996000 16188 torch/_dynamo/ex

Train - Loss: 180.6063 Accuracy: 0.7794
Valid - Loss: 9.5671 Accuracy: 1.0000
Epoch: 2
Train - Loss: 5.2181 Accuracy: 0.9982
Valid - Loss: 3.4683 Accuracy: 1.0000
Epoch: 3
Train - Loss: 1.0704 Accuracy: 1.0000
Valid - Loss: 0.7283 Accuracy: 1.0000
Epoch: 4
Train - Loss: 1.0514 Accuracy: 0.9999
Valid - Loss: 0.2899 Accuracy: 1.0000
Epoch: 5
Train - Loss: 0.3240 Accuracy: 1.0000
Valid - Loss: 0.3818 Accuracy: 1.0000
Epoch: 6
Train - Loss: 0.2134 Accuracy: 1.0000
Valid - Loss: 0.0771 Accuracy: 1.0000
Epoch: 7
Train - Loss: 0.1326 Accuracy: 1.0000
Valid - Loss: 0.0460 Accuracy: 1.0000
Epoch: 8
Train - Loss: 0.1199 Accuracy: 1.0000
Valid - Loss: 0.0659 Accuracy: 1.0000
Epoch: 9
Train - Loss: 0.1054 Accuracy: 1.0000
Valid - Loss: 0.0341 Accuracy: 1.0000
Epoch: 10
Train - Loss: 16.7315 Accuracy: 0.9770
Valid - Loss: 23.6364 Accuracy: 0.9653
Epoch: 11
Train - Loss: 2.8728 Accuracy: 0.9967
Valid - Loss: 0.0736 Accuracy: 1.0000
Epoch: 12
Train - Loss: 0.1144 Accuracy: 1.0000
Valid - Loss: 0.0304

In [17]:
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}